# Milvus functional walkthrough

This notebook mirrors the Milvus functional-test flow used by `tests/functional/test_milvus_filters.py` and the shared helpers in `tests/functional/conftest.py`, `tests/functional/data.py`, and `tests/functional/filter_assertions.py`.

## Prerequisites
- Run the notebook from this repository with Docker available.
- Expect the stack build to take time the first time it pulls images or prepares models.
- The notebook seeds the deterministic fixture documents from `tests.functional.data.DOCUMENTS`.
- For Milvus, the notebook intentionally restarts `vector-retriever` after seeding so the service re-initializes against the populated collection, matching the functional-test quirk in `conftest.py`.

## What this notebook demonstrates
1. Bring up the Milvus-backed `vector-retriever` stack with the same isolated ports used by the functional tests.
2. Wait for readiness and seed deterministic fixture data.
3. Validate `/health`, `/ready`, `/capabilities/filters`, and `/query`.
4. Exercise batch queries, `explain_filters`, `top_k` limiting, and every Milvus filter case from `FILTER_CASES` with one API call per code cell.
5. Tear the stack back down when exploration is complete.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from copy import deepcopy
from pathlib import Path
from typing import Any
from uuid import uuid4

import requests
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'docker' / 'compose.yaml').exists():
            return candidate
    raise RuntimeError('Could not locate the repository root from the notebook working directory.')


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.functional.data import DOCUMENTS, FILTER_CASES

print(f'Repository root: {REPO_ROOT}')
print(f'Loaded {len(DOCUMENTS)} fixture documents and {len(FILTER_CASES)} filter cases.')

Repository root: /home/intel/workbench/integration/lib/microservices/vector-retriever/vector-retriever
Loaded 6 fixture documents and 20 filter cases.


## Milvus-specific environment and port settings

These values intentionally mirror `PORT_MAP["milvus"]` and the backend environment assembly in `tests/functional/conftest.py`.

In [2]:
BACKEND = 'milvus'
COMPOSE_PROJECT = 'docker'
OV_MODELS_SOURCE_VOLUME = 'docker_ov_models'
OV_MODELS_DEST_VOLUME = f'{COMPOSE_PROJECT}_ov-models'
DEFAULT_TEST_EMBEDDING_MODEL_NAME = 'CLIP/clip-vit-b-32'
PORT_SETTINGS = {
    'VECTOR_RETRIEVER_HOST_PORT': '6102',
    'EMBEDDING_SERVER_PORT': '9712',
    'MILVUS_HOST_PORT': '19531',
    'MILVUS_METRICS_HOST_PORT': '9092',
}


def build_backend_env() -> dict[str, str]:
    env = os.environ.copy()
    env.update(PORT_SETTINGS)
    index_name = f'vr_functional_{BACKEND}_{uuid4().hex[:8]}'
    env['RETRIEVER_BACKEND'] = BACKEND
    env['INDEX_NAME'] = index_name
    env['VS_INDEX_NAME'] = index_name
    env['EMBEDDING_MODEL_NAME'] = env.get('EMBEDDING_MODEL_NAME', '').strip() or DEFAULT_TEST_EMBEDDING_MODEL_NAME
    env['EMBEDDING_DEVICE'] = 'CPU'
    env['EMBEDDING_USE_OV'] = 'true'
    env["VECTOR_RETRIEVER_LOG_LEVEL"] = "debug"
    return env


ENV = build_backend_env()
BASE_URL = f"http://localhost:{ENV['VECTOR_RETRIEVER_HOST_PORT']}"
display(
    JSON(
        {
            'backend': BACKEND,
            'base_url': BASE_URL,
            'ports': PORT_SETTINGS,
            'index_name': ENV['INDEX_NAME'],
            'embedding_model_name': ENV['EMBEDDING_MODEL_NAME'],
            'embedding_device': ENV['EMBEDDING_DEVICE'],
            'embedding_use_ov': ENV['EMBEDDING_USE_OV'],
        },
        expanded=False,
    )
)

<IPython.core.display.JSON object>

## Local notebook helpers

All helper functions live in this notebook so it is self-contained while still following the functional-test behavior closely.

In [3]:
def show_json(title: str, payload: Any, *, expanded: bool = False) -> None:
    display(Markdown(f'**{title}**'))
    display(JSON(payload, expanded=expanded))


def run_compose(env: dict[str, str], args: list[str]) -> subprocess.CompletedProcess:
    command = [
        'docker',
        'compose',
        '-f',
        'docker/compose.yaml',
        '-f',
        f'docker/compose.{BACKEND}.yaml',
        *args,
    ]
    print('$', ' '.join(command))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f'docker compose failed (exit {completed.returncode})\n'
            f'command: {command}\n'
            f'stdout:\n{completed.stdout}\n'
            f'stderr:\n{completed.stderr}'
        )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed


def ensure_ov_models_volume() -> None:
    if subprocess.run(['docker', 'volume', 'inspect', OV_MODELS_DEST_VOLUME], capture_output=True).returncode == 0:
        print(f'Using existing volume: {OV_MODELS_DEST_VOLUME}')
        return

    if subprocess.run(['docker', 'volume', 'inspect', OV_MODELS_SOURCE_VOLUME], capture_output=True).returncode != 0:
        print(
            'Cached OV model volume not present; the embedding service may download or prepare models at runtime.'
        )
        return

    subprocess.run(['docker', 'volume', 'create', OV_MODELS_DEST_VOLUME], check=True, capture_output=True)
    subprocess.run(
        [
            'docker',
            'run',
            '--rm',
            '-v',
            f'{OV_MODELS_SOURCE_VOLUME}:/src',
            '-v',
            f'{OV_MODELS_DEST_VOLUME}:/dst',
            'alpine',
            'sh',
            '-c',
            'cp -r /src/. /dst/',
        ],
        check=True,
        capture_output=True,
    )
    print(f'Copied cached OV models into {OV_MODELS_DEST_VOLUME}')


def wait_for_ready(base_url: str, timeout_seconds: int = 600) -> dict[str, Any]:
    start = time.time()
    last_error = ''
    while time.time() - start < timeout_seconds:
        try:
            response = requests.get(f'{base_url}/ready', timeout=10)
            if response.status_code == 200:
                payload = response.json()
                if payload.get('status') == 'ready':
                    show_json('GET /ready', payload)
                    return payload
                last_error = f'unexpected /ready payload: {payload}'
            else:
                last_error = f'/ready returned {response.status_code}: {response.text}'
        except Exception as exc:  # noqa: BLE001
            last_error = str(exc)
        print('Waiting for readiness...', last_error or 'starting')
        time.sleep(3)
    raise AssertionError(f'vector-retriever did not become ready: {last_error}')


def seed_backend_data(env: dict[str, str]) -> subprocess.CompletedProcess:
    script = '\n'.join(
        [
            'import json',
            'from src.common.settings import settings',
            'from src.retriever.backend_factory import get_vectordb',
            f'expected_backend = {BACKEND!r}',
            "assert settings.RETRIEVER_BACKEND == expected_backend, (",
            "    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'",
            ')',
            f'docs = json.loads({json.dumps(DOCUMENTS)!r})',
            'store = get_vectordb()',
            "texts = [item['page_content'] for item in docs]",
            "metadatas = [item['metadata'] for item in docs]",
            "ids = [item['metadata']['video_id'] for item in docs]",
            'try:',
            '    store.add_texts(texts=texts, metadatas=metadatas, ids=ids)',
            'except TypeError:',
            '    store.add_texts(texts=texts, metadatas=metadatas)',
            "print('seeded', len(docs))",
        ]
    )
    completed = run_compose(env, ['exec', '-T', 'vector-retriever', 'python', '-c', script])
    return completed


def api_get(path: str, *, timeout: int = 15, label: str | None = None) -> dict[str, Any]:
    response = requests.get(f'{BASE_URL}{path}', timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    show_json(label or f'GET {path}', payload)
    return payload


def summarize_item(item: dict[str, Any]) -> dict[str, Any]:
    metadata = item.get('metadata', {})
    return {
        'video_id': metadata.get('video_id'),
        'camera_zone': metadata.get('camera_zone'),
        'frame_number': metadata.get('frame_number'),
        'bucket_name': metadata.get('bucket_name'),
        'tags': metadata.get('tags'),
        'score': item.get('score'),
    }


def extract_video_ids(result: dict[str, Any]) -> list[str]:
    return sorted(
        item.get('metadata', {}).get('video_id')
        for item in result.get('items', [])
        if item.get('metadata', {}).get('video_id')
    )


def summarize_result(result: dict[str, Any]) -> dict[str, Any]:
    return {
        'query_id': result.get('query_id'),
        'count': result.get('count'),
        'matched_ids': extract_video_ids(result),
        'applied_filters': result.get('applied_filters'),
        'items': [summarize_item(item) for item in result.get('items', [])],
    }


def api_query(payload: list[dict[str, Any]], *, timeout: int = 60, label: str = 'POST /query') -> dict[str, Any]:
    response = requests.post(f'{BASE_URL}/query', json=payload, timeout=timeout)
    response.raise_for_status()
    body = response.json()
    assert not body['errors'], body['errors']
    show_json(
        label,
        {
            'request_id': body.get('request_id'),
            'errors': body.get('errors'),
            'results': [summarize_result(result) for result in body.get('results', [])],
        },
        expanded=False,
    )
    return body


def verify_seed_visible(expected_count: int) -> dict[str, Any]:
    payload = [
        {
            'query_id': 'seed-verification',
            'query': 'fixture retrieval anchor',
            'top_k': 100,
        }
    ]
    body = api_query(payload, label='Seed verification query')
    count = body['results'][0]['count']
    assert count >= expected_count, f'expected at least {expected_count} seeded docs, got {count}'
    return body['results'][0]


def run_filter_case(case: dict[str, Any]) -> dict[str, Any]:
    payload = {
        'query_id': case['name'],
        'query': 'fixture retrieval anchor',
        'top_k': 100,
        'explain_filters': True,
    }
    payload.update(deepcopy(case['payload']))
    body = api_query([payload], label=f"Filter case: {case['name']}")
    result = body['results'][0]
    matched_ids = extract_video_ids(result)
    expected_ids = sorted(case['expected_ids'])
    assert matched_ids == expected_ids, (
        f"case={case['name']} expected={expected_ids} got={matched_ids}"
    )
    print(f"Matched IDs: {matched_ids}")
    return result


def teardown_stack(env: dict[str, str] = ENV, *, remove_volumes: bool = True, ignore_errors: bool = False) -> None:
    args = ['down']
    if remove_volumes:
        args.append('-v')
    args.append('--remove-orphans')
    try:
        run_compose(env, args)
    except Exception:
        if ignore_errors:
            print('No running stack to remove or cleanup failed harmlessly.')
        else:
            raise


FILTER_CASE_LOOKUP = {case['name']: case for case in FILTER_CASES}
show_json('Fixture video IDs', [doc['metadata']['video_id'] for doc in DOCUMENTS])

**Fixture video IDs**

<IPython.core.display.JSON object>

## Optional reset

If a previous notebook run left the Milvus stack up, run this cleanup cell before starting fresh.

In [4]:
teardown_stack(ignore_errors=True)

$ docker compose -f docker/compose.yaml -f docker/compose.milvus.yaml down -v --remove-orphans
time="2026-05-25T11:28:35+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
 Volume docker_milvus-etcd Removing 
 Volume docker_milvus-db Removing 
 Volume docker_milvus-etcd Removed 
 Volume docker_milvus-db Removed


## Start the Milvus functional stack

This uses the same compose files as the functional suite: `docker/compose.yaml` plus `docker/compose.milvus.yaml`.

In [5]:
ensure_ov_models_volume()
run_compose(ENV, ['up', '-d', '--build'])
run_compose(ENV, ['ps'])

Copied cached OV models into docker_ov-models
$ docker compose -f docker/compose.yaml -f docker/compose.milvus.yaml up -d --build
#1 [internal] load local bake definitions
#1 reading from stdin 936B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.10kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.12.13-slim
#3 DONE 1.1s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/8] FROM docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203
#5 resolve docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203 0.0s done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 6.02kB done
#6 DONE 0.0s

#7 [2/8] RUN groupadd --gid 1000 appuser &&     useradd --uid 1000 --gid appuser --shell /bin/bash --create-home appuser
#7 CACHED

#8 [3/8] WORKDIR /app
#8 CA

CompletedProcess(args=['docker', 'compose', '-f', 'docker/compose.yaml', '-f', 'docker/compose.milvus.yaml', 'ps'], returncode=0, stdout='NAME                           IMAGE                                       COMMAND                  SERVICE                        CREATED          STATUS                                     PORTS\ndocker-milvus-etcd-1           quay.io/coreos/etcd:v3.5.18                 "etcd -advertise-cli…"   milvus-etcd                    32 seconds ago   Up 30 seconds (healthy)                    2379-2380/tcp\ndocker-milvus-standalone-1     milvusdb/milvus:v2.6.14                     "/tini -- milvus run…"   milvus-standalone              31 seconds ago   Up Less than a second (health: starting)   0.0.0.0:9092->9091/tcp, [::]:9092->9091/tcp, 0.0.0.0:19531->19530/tcp, [::]:19531->19530/tcp\ndocker-vector-retriever-1      vector-retriever-milvus:latest              "/app/scripts/entryp…"   vector-retriever               31 seconds ago   Up Less than a second    

## Wait for service readiness

Poll `/ready` until `vector-retriever` reports `status=ready`.

In [6]:
wait_for_ready(BASE_URL)

Waiting for readiness... ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


**GET /ready**

<IPython.core.display.JSON object>

{'status': 'ready', 'timestamp': '2026-05-25T05:59:46.327339Z'}

## Seed deterministic fixture documents

The seeded documents come directly from `tests.functional.data.DOCUMENTS`. After inserting them, restart `vector-retriever` so Milvus re-initializes against the populated collection, matching `tests/functional/conftest.py`.

In [7]:
seed_backend_data(ENV)
run_compose(ENV, ['restart', 'vector-retriever'])
wait_for_ready(BASE_URL)

$ docker compose -f docker/compose.yaml -f docker/compose.milvus.yaml exec -T vector-retriever python -c import json
from src.common.settings import settings
from src.retriever.backend_factory import get_vectordb
expected_backend = 'milvus'
assert settings.RETRIEVER_BACKEND == expected_backend, (
    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'
)
docs = json.loads('[{"page_content": "red car at intersection", "metadata": {"video_id": "vid-001", "camera_zone": "north", "description": "red car at intersection", "prefix": "alpha-route", "frame_number": 10, "bucket_name": "cam-a", "tags": ["traffic", "vehicle", "red"], "created_at": "2026-03-10T10:00:00+00:00", "optional_note": "doc has note"}}, {"page_content": "blue bus near station", "metadata": {"video_id": "vid-002", "camera_zone": "south", "description": "blue bus near station", "prefix": "beta-route", "frame_number": 20, "bucket_name": "cam-b", "tags": ["traffic", "vehicle", "bus"], "created

**GET /ready**

<IPython.core.display.JSON object>

{'status': 'ready', 'timestamp': '2026-05-25T05:59:54.402363Z'}

## Verify the seed is visible

Before exploring filters, make sure a broad query can see all seeded fixture records.

In [8]:
verify_seed_visible(expected_count=len(DOCUMENTS))

**Seed verification query**

<IPython.core.display.JSON object>

{'query_id': 'seed-verification',
 'query': 'fixture retrieval anchor',
 'count': 6,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_

## Endpoint check: `/health`

Confirm the service liveness endpoint returns `status="ok"`.

In [9]:
health_payload = api_get('/health')
assert health_payload['status'] == 'ok', health_payload
health_payload

**GET /health**

<IPython.core.display.JSON object>

{'status': 'ok', 'timestamp': '2026-05-25T05:59:54.446666Z'}

## Endpoint check: `/ready`

Confirm the ready endpoint still reports `status="ready"` after seeding and restart.

In [10]:
ready_payload = api_get('/ready')
assert ready_payload['status'] == 'ready', ready_payload
ready_payload

**GET /ready**

<IPython.core.display.JSON object>

{'status': 'ready', 'timestamp': '2026-05-25T05:59:54.455594Z'}

## Endpoint check: `/capabilities/filters`

Inspect the active backend and the advertised filter-capability surface.

In [11]:
capabilities_payload = api_get('/capabilities/filters')
assert capabilities_payload['active_backend'] == BACKEND, capabilities_payload
advertised_backends = {backend['backend'] for backend in capabilities_payload.get('backends', [])}
assert BACKEND in advertised_backends, advertised_backends
capabilities_payload

**GET /capabilities/filters**

<IPython.core.display.JSON object>

{'active_backend': 'milvus',
 'backends': [{'backend': 'faiss',
   'top_level_fields': ['query', 'top_k', 'where'],
   'logical_blocks': ['all', 'any', 'not'],
   'supported_operators': ['eq',
    'in',
    'contains',
    'starts_with',
    'gt',
    'gte',
    'lt',
    'lte',
    'between',
    'contains_any',
    'contains_all',
    'exists',
    'missing'],
   'pushdown_operators': ['eq', 'in', 'gte', 'lte', 'between'],
   'known_fields': {'tags': 'array<string>', 'created_at': 'datetime'},
   'max_where_depth': 5,
   'max_where_clauses': 50,
   'max_where_list_size': 100},
  {'backend': 'milvus',
   'top_level_fields': ['query', 'top_k', 'where'],
   'logical_blocks': ['all', 'any', 'not'],
   'supported_operators': ['eq',
    'in',
    'contains',
    'starts_with',
    'gt',
    'gte',
    'lt',
    'lte',
    'between',
    'contains_any',
    'contains_all',
    'exists',
    'missing'],
   'pushdown_operators': ['eq', 'in', 'gte', 'lte', 'between'],
   'known_fields': {'tags

## Endpoint check: `/query`

Run a simple similarity query against the seeded collection before moving into the specialized filter checks.

In [12]:
smoke_query_body = api_query(
    [
        {
            'query_id': 'smoke-query',
            'query': 'fixture retrieval anchor',
            'top_k': 3,
        }
    ],
    label='Query smoke test',
)
assert smoke_query_body['results'][0]['count'] >= 3, smoke_query_body
smoke_query_body['results'][0]

**Query smoke test**

<IPython.core.display.JSON object>

{'query_id': 'smoke-query',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'prefix': 'delta-night',
    'frame_number': 60,
    'bucket_name': 'cam-e',
    'tags': ['night', 'traffic'],
    'created_at': '2026-04-05T20:10:00+00:00'},
   'page_content': 'empty intersection at night'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
 

## Batch query

Mirror `assert_batch_query` by issuing two independent queries in a single `/query` call and checking that each response block returns the expected document.

In [13]:
batch_query_body = api_query(
    [
        {
            'query_id': 'batch-q1',
            'query': 'fixture retrieval anchor',
            'where': {'field': 'video_id', 'op': 'eq', 'value': 'vid-001'},
            'top_k': 10,
        },
        {
            'query_id': 'batch-q2',
            'query': 'fixture retrieval anchor',
            'where': {'field': 'video_id', 'op': 'eq', 'value': 'vid-002'},
            'top_k': 10,
        },
    ],
    label='Batch query check',
)
ids_q1 = extract_video_ids(batch_query_body['results'][0])
ids_q2 = extract_video_ids(batch_query_body['results'][1])
assert ids_q1 == ['vid-001'], ids_q1
assert ids_q2 == ['vid-002'], ids_q2
batch_query_body['results']

**Batch query check**

<IPython.core.display.JSON object>

[{'query_id': 'batch-q1',
  'query': 'fixture retrieval anchor',
  'count': 1,
  'items': [{'score': 1.4777005910873413,
    'metadata': {'pk': 'vid-001',
     'video_id': 'vid-001',
     'camera_zone': 'north',
     'description': 'red car at intersection',
     'prefix': 'alpha-route',
     'frame_number': 10,
     'bucket_name': 'cam-a',
     'tags': ['traffic', 'vehicle', 'red'],
     'created_at': '2026-03-10T10:00:00+00:00',
     'optional_note': 'doc has note'},
    'page_content': 'red car at intersection'}],
  'applied_filters': {'tags': None,
   'time_filter': None,
   'filters': None,
   'normalized_where': {'field': 'video_id',
    'op': 'eq',
    'value': 'vid-001',
    'all': None,
    'any': None,
    'not': None},
   'warnings': None,
   'compiled_backend_filter': None,
   'dropped_or_rewritten_clauses': None}},
 {'query_id': 'batch-q2',
  'query': 'fixture retrieval anchor',
  'count': 1,
  'items': [{'score': 1.4708623886108398,
    'metadata': {'pk': 'vid-002',
     

## `explain_filters`

Mirror `assert_explain_filters` and confirm the response includes a backend-compiled filter description when `explain_filters=True`.

In [14]:
explain_filters_body = api_query(
    [
        {
            'query_id': 'explain-test',
            'query': 'fixture retrieval anchor',
            'where': {'field': 'frame_number', 'op': 'gte', 'value': 50},
            'top_k': 10,
            'explain_filters': True,
        }
    ],
    label='Explain filters check',
)
applied_filters = explain_filters_body['results'][0]['applied_filters']
assert applied_filters.get('compiled_backend_filter') is not None, applied_filters
applied_filters

**Explain filters check**

<IPython.core.display.JSON object>

{'tags': None,
 'time_filter': None,
 'filters': None,
 'normalized_where': {'field': 'frame_number',
  'op': 'gte',
  'value': 50,
  'all': None,
  'any': None,
  'not': None},
 'warnings': None,
 'compiled_backend_filter': 'frame_number >= 50',
 'dropped_or_rewritten_clauses': []}

## `top_k` limiting

Mirror `assert_top_k_limiting` and verify the service returns exactly two results when `top_k=2`.

In [15]:
top_k_body = api_query(
    [
        {
            'query_id': 'topk-limit',
            'query': 'fixture retrieval anchor',
            'top_k': 2,
        }
    ],
    label='Top-k limiting check',
)
top_k_result = top_k_body['results'][0]
assert top_k_result['count'] == 2, top_k_result
assert len(top_k_result['items']) == 2, top_k_result
top_k_result

**Top-k limiting check**

<IPython.core.display.JSON object>

{'query_id': 'topk-limit',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'prefix': 'alpha-signal',
    'frame_number': 50,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'signal'],
    'created_at': '2026-04-01T09:45:00+00:00',
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': None,
  'time_filter': None,


## Image query

This demonstrates the image query modality introduced alongside the existing text query.
An image can be provided as `image_base64` or `image_url` instead of a text `query`.
The two modalities are mutually exclusive — providing both returns a `422` validation error.

The test uses a tiny 1×1 red PNG encoded in base64 so it works without network access.

In [16]:
RED_1X1_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR4nGP4"
    "z8BQDwAEgAF/pooBPQAAAABJRU5ErkJggg=="
)

# --- image_base64 query ---
image_body = api_query(
    [
        {
            "query_id": "image-base64-test",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
            "top_k": 5,
        }
    ],
    label="Image base64 query",
)
image_result = image_body["results"][0]
assert image_result["query_id"] == "image-base64-test"
assert image_result["query"] == "[image_base64]", image_result["query"]
assert image_result["count"] >= 1, image_result["count"]
show_json("Image query result", image_result)

# --- image query with where filter ---
filtered_body = api_query(
    [
        {
            "query_id": "image-filtered-test",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
            "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
            "top_k": 10,
        }
    ],
    label="Image query with where filter",
)
filtered_result = filtered_body["results"][0]
matched_ids = {item["metadata"]["video_id"] for item in filtered_result["items"]}
assert matched_ids <= {"vid-001"}, matched_ids
show_json("Filtered image query result", filtered_result)

# --- mutual exclusivity: both query and image should fail ---
invalid_response = requests.post(
    f"{BASE_URL}/query",
    json=[{"query": "test", "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64}}],
    timeout=60,
)
assert invalid_response.status_code == 422, invalid_response.status_code
print(f"Mutual exclusivity correctly rejected with {invalid_response.status_code}")


**Image base64 query**

<IPython.core.display.JSON object>

**Image query result**

<IPython.core.display.JSON object>

**Image query with where filter**

<IPython.core.display.JSON object>

**Filtered image query result**

<IPython.core.display.JSON object>

Mutual exclusivity correctly rejected with 422


## Filter-case walkthroughs

The remaining cells mirror the parametrized `test_milvus_filter` cases. Each code cell below makes one `/query` request and validates the exact expected video IDs.

### Filter case: `op_eq`

            Exact-match filtering on a single scalar metadata field.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-001"
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001"
]
            ```

In [17]:
run_filter_case(FILTER_CASE_LOOKUP['op_eq'])

**Filter case: op_eq**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001']


{'query_id': 'op_eq',
 'query': 'fixture retrieval anchor',
 'count': 1,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'video_id',
   'op': 'eq',
   'value': 'vid-001',
   'all': None,
   'any': None,
   'not': None},
  'warnings': None,
  'compiled_backend_filter': 'video_id == "vid-001"',
  'dropped_or_rewritten_clauses': []}}

### Filter case: `op_in`

            Membership filtering against a list of allowed values.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "camera_zone",
    "op": "in",
    "value": [
      "north",
      "east"
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-003"
]
            ```

In [18]:
run_filter_case(FILTER_CASE_LOOKUP['op_in'])

**Filter case: op_in**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-003']


{'query_id': 'op_in',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filt

### Filter case: `op_contains`

            Substring matching against free-text metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "description",
    "op": "contains",
    "value": "intersection"
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-006"
]
            ```

In [19]:
run_filter_case(FILTER_CASE_LOOKUP['op_contains'])

**Filter case: op_contains**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-006']


{'query_id': 'op_contains',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'prefix': 'delta-night',
    'frame_number': 60,
    'bucket_name': 'cam-e',
    'tags': ['night', 'traffic'],
    'created_at': '2026-04-05T20:10:00+00:00'},
   'page_content': 'empty intersection at night'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {

### Filter case: `op_starts_with`

            Prefix matching for string metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "prefix",
    "op": "starts_with",
    "value": "alpha"
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-003",
  "vid-005"
]
            ```

In [20]:
run_filter_case(FILTER_CASE_LOOKUP['op_starts_with'])

**Filter case: op_starts_with**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-003', 'vid-005']


{'query_id': 'op_starts_with',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vi

### Filter case: `op_gt`

            Strict greater-than filtering for numeric metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "gt",
    "value": 40
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-005",
  "vid-006"
]
            ```

In [21]:
run_filter_case(FILTER_CASE_LOOKUP['op_gt'])

**Filter case: op_gt**

<IPython.core.display.JSON object>

Matched IDs: ['vid-005', 'vid-006']


{'query_id': 'op_gt',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'prefix': 'delta-night',
    'frame_number': 60,
    'bucket_name': 'cam-e',
    'tags': ['night', 'traffic'],
    'created_at': '2026-04-05T20:10:00+00:00'},
   'page_content': 'empty intersection at night'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'prefix': 'alpha-signal',
    'frame_number': 50,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'signal'],
    'created_at': '2026-04-01T09:45:00+00:00',
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'fie

### Filter case: `op_gte`

            Greater-than-or-equal filtering for numeric metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "gte",
    "value": 40
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-004",
  "vid-005",
  "vid-006"
]
            ```

In [22]:
run_filter_case(FILTER_CASE_LOOKUP['op_gte'])

**Filter case: op_gte**

<IPython.core.display.JSON object>

Matched IDs: ['vid-004', 'vid-005', 'vid-006']


{'query_id': 'op_gte',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'prefix': 'delta-night',
    'frame_number': 60,
    'bucket_name': 'cam-e',
    'tags': ['night', 'traffic'],
    'created_at': '2026-04-05T20:10:00+00:00'},
   'page_content': 'empty intersection at night'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'prefix': 'alpha-signal',
    'frame_number': 50,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'signal'],
    'created_at': '2026-04-01T09:45:00+00:00',
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'},
  {'score': 0.9070409536361694,
   'metadata': {'pk': 'vid-004',
    'video_id': 'vid-004',
    'camera_

### Filter case: `op_lt`

            Strict less-than filtering for numeric metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "lt",
    "value": 30
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-002"
]
            ```

In [23]:
run_filter_case(FILTER_CASE_LOOKUP['op_lt'])

**Filter case: op_lt**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-002']


{'query_id': 'op_lt',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'frame

### Filter case: `op_lte`

            Less-than-or-equal filtering for numeric metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "lte",
    "value": 30
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-002",
  "vid-003"
]
            ```

In [24]:
run_filter_case(FILTER_CASE_LOOKUP['op_lte'])

**Filter case: op_lte**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-002', 'vid-003']


{'query_id': 'op_lte',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'eas

### Filter case: `op_between`

            Inclusive numeric range filtering.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "between",
    "value": [
      20,
      40
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-002",
  "vid-003",
  "vid-004"
]
            ```

In [25]:
run_filter_case(FILTER_CASE_LOOKUP['op_between'])

**Filter case: op_between**

<IPython.core.display.JSON object>

Matched IDs: ['vid-002', 'vid-003', 'vid-004']


{'query_id': 'op_between',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9070409536361694,
   'metadata': {'pk': 'vid-004',
    'video_id': 'vid-004',
    'camera

### Filter case: `op_contains_any`

            Array overlap filtering that matches any requested tag.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "tags",
    "op": "contains_any",
    "value": [
      "bicycle",
      "signal"
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-003",
  "vid-005"
]
            ```

In [26]:
run_filter_case(FILTER_CASE_LOOKUP['op_contains_any'])

**Filter case: op_contains_any**

<IPython.core.display.JSON object>

Matched IDs: ['vid-003', 'vid-005']


{'query_id': 'op_contains_any',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'prefix': 'alpha-signal',
    'frame_number': 50,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'signal'],
    'created_at': '2026-04-01T09:45:00+00:00',
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': None,
  'time_filter': N

### Filter case: `op_contains_all`

            Array containment filtering that requires every requested tag.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "tags",
    "op": "contains_all",
    "value": [
      "traffic",
      "vehicle"
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-002"
]
            ```

In [27]:
run_filter_case(FILTER_CASE_LOOKUP['op_contains_all'])

**Filter case: op_contains_all**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-002']


{'query_id': 'op_contains_all',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'fiel

### Filter case: `op_exists`

            Field-presence filtering for optional metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "optional_note",
    "op": "exists"
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-003",
  "vid-005"
]
            ```

In [28]:
run_filter_case(FILTER_CASE_LOOKUP['op_exists'])

**Filter case: op_exists**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-003', 'vid-005']


{'query_id': 'op_exists',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005

### Filter case: `op_missing`

            Field-absence filtering for optional metadata.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "optional_note",
    "op": "missing"
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-002",
  "vid-004",
  "vid-006"
]
            ```

In [29]:
run_filter_case(FILTER_CASE_LOOKUP['op_missing'])

**Filter case: op_missing**

<IPython.core.display.JSON object>

Matched IDs: ['vid-002', 'vid-004', 'vid-006']


{'query_id': 'op_missing',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'prefix': 'delta-night',
    'frame_number': 60,
    'bucket_name': 'cam-e',
    'tags': ['night', 'traffic'],
    'created_at': '2026-04-05T20:10:00+00:00'},
   'page_content': 'empty intersection at night'},
  {'score': 0.9070409536361694,
   'metadata': {'pk': 'vid-004',
    'video_id': 'vid-004',
    'camera_zone': 'west',
    'description': 'deliv

### Filter case: `logical_compound`

            Nested `all`/`any`/`not` logic composed into one request.

            **Request fragment**
            ```json
            {
  "where": {
    "all": [
      {
        "field": "frame_number",
        "op": "gte",
        "value": 20
      },
      {
        "any": [
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "east"
          },
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "west"
          }
        ]
      },
      {
        "not": {
          "field": "tags",
          "op": "contains_any",
          "value": [
            "night"
          ]
        }
      }
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-003",
  "vid-004"
]
            ```

In [30]:
run_filter_case(FILTER_CASE_LOOKUP['logical_compound'])

**Filter case: logical_compound**

<IPython.core.display.JSON object>

Matched IDs: ['vid-003', 'vid-004']


{'query_id': 'logical_compound',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9070409536361694,
   'metadata': {'pk': 'vid-004',
    'video_id': 'vid-004',
    'camera_zone': 'west',
    'description': 'delivery truck unloading',
    'prefix': 'gamma-log',
    'frame_number': 40,
    'bucket_name': 'cam-d',
    'tags': ['logistics', 'vehicle'],
    'created_at': '2026-03-25T14:15:00+00:00'},
   'page_content': 'delivery truck unloading'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where'

### Filter case: `legacy_tags_and_time_filter`

            Backward-compatible legacy `tags` plus `time_filter` request shape.

            **Request fragment**
            ```json
            {
  "tags": [
    "traffic"
  ],
  "time_filter": {
    "end": "2026-04-02T00:00:00+00:00",
    "start": "2026-03-12T00:00:00+00:00"
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-002",
  "vid-005"
]
            ```

In [31]:
run_filter_case(FILTER_CASE_LOOKUP['legacy_tags_and_time_filter'])

**Filter case: legacy_tags_and_time_filter**

<IPython.core.display.JSON object>

Matched IDs: ['vid-002', 'vid-005']


{'query_id': 'legacy_tags_and_time_filter',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'prefix': 'alpha-signal',
    'frame_number': 50,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'signal'],
    'created_at': '2026-04-01T09:45:00+00:00',
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': ['traffic'],
  'time_filter': {'start': '2026-03-12T00:00:00Z'

### Filter case: `legacy_filters_map`

            Backward-compatible legacy `filters` map request shape.

            **Request fragment**
            ```json
            {
  "filters": {
    "bucket_name": {
      "op": "in",
      "value": [
        "cam-a",
        "cam-b"
      ]
    },
    "frame_number": {
      "op": "between",
      "value": [
        10,
        20
      ]
    }
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-001",
  "vid-002"
]
            ```

In [32]:
run_filter_case(FILTER_CASE_LOOKUP['legacy_filters_map'])

**Filter case: legacy_filters_map**

<IPython.core.display.JSON object>

Matched IDs: ['vid-001', 'vid-002']


{'query_id': 'legacy_filters_map',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 1.4777005910873413,
   'metadata': {'pk': 'vid-001',
    'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': {'bucket_name': {'op': 'in', 'v

### Filter case: `logical_not_toplevel`

            Top-level logical negation without extra nesting.

            **Request fragment**
            ```json
            {
  "where": {
    "not": {
      "field": "camera_zone",
      "op": "eq",
      "value": "north"
    }
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-002",
  "vid-003",
  "vid-004",
  "vid-005",
  "vid-006"
]
            ```

In [33]:
run_filter_case(FILTER_CASE_LOOKUP['logical_not_toplevel'])

**Filter case: logical_not_toplevel**

<IPython.core.display.JSON object>

Matched IDs: ['vid-002', 'vid-003', 'vid-004', 'vid-005', 'vid-006']


{'query_id': 'logical_not_toplevel',
 'query': 'fixture retrieval anchor',
 'count': 5,
 'items': [{'score': 1.4708623886108398,
   'metadata': {'pk': 'vid-002',
    'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 1.2737712860107422,
   'metadata': {'pk': 'vid-006',
    'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'prefix': 'delta-night',
    'frame_number': 60,
    'bucket_name': 'cam-e',
    'tags': ['night', 'traffic'],
    'created_at': '2026-04-05T20:10:00+00:00'},
   'page_content': 'empty intersection at night'},
  {'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'descriptio

### Filter case: `no_match`

            A negative control that should return an empty result set.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-999"
  }
}
            ```

            **Expected video IDs**
            ```json
            []
            ```

In [34]:
run_filter_case(FILTER_CASE_LOOKUP['no_match'])

**Filter case: no_match**

<IPython.core.display.JSON object>

Matched IDs: []


{'query_id': 'no_match',
 'query': 'fixture retrieval anchor',
 'count': 0,
 'items': [],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'video_id',
   'op': 'eq',
   'value': 'vid-999',
   'all': None,
   'any': None,
   'not': None},
  'warnings': None,
  'compiled_backend_filter': 'video_id == "vid-999"',
  'dropped_or_rewritten_clauses': []}}

### Filter case: `op_time_between_where`

            Datetime range filtering expressed through the modern `where` grammar.

            **Request fragment**
            ```json
            {
  "where": {
    "field": "created_at",
    "op": "between",
    "value": [
      "2026-03-20T00:00:00+00:00",
      "2026-04-01T23:59:59+00:00"
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-003",
  "vid-004",
  "vid-005"
]
            ```

In [35]:
run_filter_case(FILTER_CASE_LOOKUP['op_time_between_where'])

**Filter case: op_time_between_where**

<IPython.core.display.JSON object>

Matched IDs: ['vid-003', 'vid-004', 'vid-005']


{'query_id': 'op_time_between_where',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.9245074391365051,
   'metadata': {'pk': 'vid-005',
    'video_id': 'vid-005',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'prefix': 'alpha-signal',
    'frame_number': 50,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'signal'],
    'created_at': '2026-04-01T09:45:00+00:00',
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'},
  {'score': 0.9070409536361694,
   'metadata': {

### Filter case: `nested_all`

            Nested `all` groups to prove recursive boolean composition works.

            **Request fragment**
            ```json
            {
  "where": {
    "all": [
      {
        "all": [
          {
            "field": "frame_number",
            "op": "gte",
            "value": 10
          },
          {
            "field": "frame_number",
            "op": "lte",
            "value": 30
          }
        ]
      },
      {
        "field": "bucket_name",
        "op": "eq",
        "value": "cam-c"
      }
    ]
  }
}
            ```

            **Expected video IDs**
            ```json
            [
  "vid-003"
]
            ```

In [36]:
run_filter_case(FILTER_CASE_LOOKUP['nested_all'])

**Filter case: nested_all**

<IPython.core.display.JSON object>

Matched IDs: ['vid-003']


{'query_id': 'nested_all',
 'query': 'fixture retrieval anchor',
 'count': 1,
 'items': [{'score': 1.1559438705444336,
   'metadata': {'pk': 'vid-003',
    'video_id': 'vid-003',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'prefix': 'alpha-bike',
    'frame_number': 30,
    'bucket_name': 'cam-c',
    'tags': ['pedestrian', 'bicycle'],
    'created_at': '2026-03-20T08:30:00+00:00',
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': None,
   'op': None,
   'value': None,
   'all': [{'field': None,
     'op': None,
     'value': None,
     'all': [{'field': 'frame_number',
       'op': 'gte',
       'value': 10,
       'all': None,
       'any': None,
       'not': None},
      {'field': 'frame_number',
       'op': 'lte',
       'value': 30,
       'all': None,
       'any': None,
       'not': None

## Final teardown

When you are done exploring, tear down the compose stack and remove its volumes so the notebook leaves the environment clean.

In [37]:
teardown_stack()

$ docker compose -f docker/compose.yaml -f docker/compose.milvus.yaml down -v --remove-orphans


time="2026-05-25T11:29:57+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
 Container docker-vector-retriever-1 Stopping 
 Container docker-vector-retriever-1 Stopped 
 Container docker-vector-retriever-1 Removing 
 Container docker-vector-retriever-1 Removed 
 Container docker-milvus-standalone-1 Stopping 
 Container multimodal-embedding-serving Stopping 
 Container multimodal-embedding-serving Stopped 
 Container multimodal-embedding-serving Removing 
 Container multimodal-embedding-serving Removed 
 Container docker-milvus-standalone-1 Stopped 
 Container docker-milvus-standalone-1 Removing 
 Container docker-milvus-standalone-1 Removed 
 Container docker-milvus-etcd-1 Stopping 
 Container docker-milvus-etcd-1 Stopped 
 Container docker-milvus-etcd-1 Removing 
 Container docker-milvus-etcd-1 Removed 
 Volume docker_data-prep Removing 
 Volume docker_milvus-etcd Removing 
 Volume docker_ov-models Removing 
 Volume docker_milvus-db Removin